# Pegasus — Laboratório de Análise Quantitativa

Este notebook implementa o pipeline completo de validação institucional:

1. **Navalha de Occam** — 2 features (Hurst + Shannon) vs 10 features  
2. **Feature Importance com SHAP** — descobre quais filtros realmente importam  
3. **Validação Walk-Forward (WFO)** — validação temporal correta para série de tempo  
4. **Métricas Robustas** — Expectância Matemática, correlação serial, MDD  
5. **Otimização do take_profit_percent** — encontra o sweet spot de saída  

**Pré-requisito:** `data/shadow_ticks.csv` com pelo menos 500 linhas (idealmente 1500+).

In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ML stack
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline

import xgboost as xgb
import shap

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 110

SHADOW_CSV = Path('data/shadow_ticks.csv')
assert SHADOW_CSV.exists(), f'Arquivo não encontrado: {SHADOW_CSV}. Rode shadow_collect.py primeiro.'

df = pd.read_csv(SHADOW_CSV)
print(f'Linhas: {len(df):,} | Colunas: {df.columns.tolist()}')

In [ ]:
# ---------------------------------------------------------------------------
# Preparação dos Dados
# ---------------------------------------------------------------------------

FEATURE_COLS = [
    'hurst_exponent',
    'shannon_entropy',
    'tick_imbalance',
    'hawkes_intensity',
    'velocity_zscore',
    'acceleration_zscore',
    'kalman_residual_zscore',
    'pmi_distance_percent',
    'markov_p_up_given_up',
    'markov_p_down_given_down',
    'bb_width_percent',
    'tick_atr_percent',
    'recent_move_percent',
]
OCCAM_COLS = ['hurst_exponent', 'shannon_entropy']  # Navalha de Occam
TARGET = 'future_result'

# Filtra apenas registros com resultado resolvido
df_clean = df.dropna(subset=FEATURE_COLS + [TARGET]).copy()
df_clean = df_clean[df_clean[TARGET].isin(['WIN', 'LOSS'])].copy()
df_clean['y'] = (df_clean[TARGET] == 'WIN').astype(int)
df_clean = df_clean.sort_values('entry_epoch').reset_index(drop=True)

print(f'Amostras limpas: {len(df_clean):,}')
print(f'WIN: {df_clean["y"].sum():,} | LOSS: {(1-df_clean["y"]).sum():,}')
print(f'Win rate bruto: {df_clean["y"].mean()*100:.1f}%')

if len(df_clean) < 200:
    print('⚠️  AMOSTRA MUITO PEQUENA (<200). Colete mais ticks antes de tirar conclusões.')

## 1. Métricas Robustas: Expectância Matemática e Correlação Serial

In [ ]:
# ---------------------------------------------------------------------------
# Expectância Matemática por Trade
# ---------------------------------------------------------------------------

if 'future_max_move_percent' in df_clean.columns:
    wins = df_clean[df_clean['y'] == 1]
    losses = df_clean[df_clean['y'] == 0]

    # Aproxima P&L: WIN = take_profit_percent (default 3%), LOSS = stake (1x)
    # Em ACCU 3%: 1 loss cancela ~33 wins com stake idêntico
    stake = 1.0
    win_pct = 0.03   # 3% take profit → ~0.03 de retorno sobre stake
    loss_pct = 1.0   # perde todo o stake

    win_rate = df_clean['y'].mean()
    avg_win = win_pct * stake
    avg_loss = loss_pct * stake

    expectancy = win_rate * avg_win - (1 - win_rate) * avg_loss

    print('=== Expectância Matemática ===')
    print(f'Win Rate:     {win_rate*100:.2f}%')
    print(f'Avg Win (est): R$ {avg_win:.4f}')
    print(f'Avg Loss (est): R$ {avg_loss:.4f}')
    print(f'Expectância por trade: R$ {expectancy:.6f}')
    print(f'Break-even win rate: {loss_pct/(win_pct+loss_pct)*100:.1f}%')
    print()
    if expectancy > 0:
        print('✅  Expectância POSITIVA (Edge existe)')
    else:
        print('❌  Expectância NEGATIVA (Edge inexistente neste dataset)')

In [ ]:
# ---------------------------------------------------------------------------
# Correlação Serial dos Retornos
# ---------------------------------------------------------------------------
# Testa se vitórias chegam em blocos (clustering) ou são independentes

from scipy import stats

returns = df_clean['y'].values.astype(float)
lag1_corr, p_value = stats.pearsonr(returns[:-1], returns[1:])

print('=== Correlação Serial (lag-1) ===')
print(f'Correlação: {lag1_corr:.4f}')
print(f'p-value:    {p_value:.4f}')
if p_value < 0.05:
    print('⚠️  Correlação serial SIGNIFICATIVA: vitórias não são independentes (clustering).')
    print('   → O cooldown dinâmico baseado em Hurst é crítico para este sistema.')
else:
    print('✅  Sem correlação serial significativa: resultados são aproximadamente i.i.d.')

## 2. SHAP Feature Importance — Quais filtros realmente importam?

In [ ]:
# ---------------------------------------------------------------------------
# Treina XGBoost e calcula SHAP values
# ---------------------------------------------------------------------------

X = df_clean[FEATURE_COLS].to_numpy(dtype=float)
y = df_clean['y'].to_numpy()

# Usa os primeiros 80% para treino (sem data leakage)
split = int(len(X) * 0.80)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

if len(y_test) > 0 and len(np.unique(y_test)) > 1:
    proba = xgb_model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    print(f'AUC-ROC (hold-out): {auc:.4f}')
    print(classification_report(y_test, xgb_model.predict(X_test), target_names=['LOSS', 'WIN']))
else:
    print('⚠️  Hold-out com apenas uma classe. Colete mais dados.')

In [ ]:
# SHAP summary plot
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_train)

plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_train,
    feature_names=FEATURE_COLS,
    show=False,
    plot_type='bar',
)
plt.title('SHAP Feature Importance — Contribuição para P(WIN)')
plt.tight_layout()
plt.savefig('logs/shap_importance.png', dpi=110)
plt.show()
print('Salvo em logs/shap_importance.png')

## 3. Navalha de Occam — 2 Features vs 10+ Features

In [ ]:
# ---------------------------------------------------------------------------
# Experimento: apenas Hurst + Shannon vs todos os indicadores
# Comparação via AUC-ROC no hold-out
# ---------------------------------------------------------------------------

from sklearn.model_selection import cross_val_score, TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

pipe_full = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
])
pipe_occam = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
])

X_full = df_clean[FEATURE_COLS].to_numpy(dtype=float)
X_occam = df_clean[OCCAM_COLS].to_numpy(dtype=float)
y_all = df_clean['y'].to_numpy()

# Cross-validation temporal (WFO simplificado)
auc_full = cross_val_score(pipe_full, X_full, y_all, cv=tscv, scoring='roc_auc').mean()
auc_occam = cross_val_score(pipe_occam, X_occam, y_all, cv=tscv, scoring='roc_auc').mean()

print('=== Navalha de Occam ===')
print(f'AUC 10+ features (todos):     {auc_full:.4f}')
print(f'AUC  2 features  (Hurst+SHA): {auc_occam:.4f}')
print()
if auc_occam >= auc_full - 0.02:
    print('✅  Navalha de Occam CONFIRMADA: 2 features performam tão bem quanto 10+.')
    print('   → Reduza o modelo para Hurst + Shannon.')
else:
    print('❌  Navalha de Occam REFUTADA: features extras adicionam sinal real.')
    print('   → Use o SHAP acima para escolher os top-3 features.')

## 4. Validação Walk-Forward (WFO) — Janelas Deslizantes

In [ ]:
# ---------------------------------------------------------------------------
# WFO: treina em janelas passadas, valida no futuro imediato
# Detecta concept drift
# ---------------------------------------------------------------------------

MIN_TRAIN = 200  # mínimo de amostras para treinar
VAL_WINDOW = 50  # amostras por janela de validação

results_wfo = []

n = len(df_clean)
step = VAL_WINDOW

for end_train in range(MIN_TRAIN, n - VAL_WINDOW + 1, step):
    train_idx = slice(0, end_train)
    val_idx = slice(end_train, end_train + VAL_WINDOW)

    X_tr = df_clean.iloc[train_idx][FEATURE_COLS].to_numpy(dtype=float)
    y_tr = df_clean.iloc[train_idx]['y'].to_numpy()
    X_va = df_clean.iloc[val_idx][FEATURE_COLS].to_numpy(dtype=float)
    y_va = df_clean.iloc[val_idx]['y'].to_numpy()

    if len(np.unique(y_tr)) < 2 or len(y_va) == 0:
        continue

    scaler = StandardScaler()
    clf = LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)
    clf.fit(scaler.fit_transform(X_tr), y_tr)

    if len(np.unique(y_va)) > 1:
        proba_va = clf.predict_proba(scaler.transform(X_va))[:, 1]
        auc_va = roc_auc_score(y_va, proba_va)
    else:
        auc_va = float('nan')

    val_epoch = df_clean.iloc[end_train]['entry_epoch']
    results_wfo.append({
        'val_start_epoch': val_epoch,
        'train_size': end_train,
        'val_winrate': y_va.mean(),
        'val_auc': auc_va,
    })

wfo_df = pd.DataFrame(results_wfo)
print(f'Janelas WFO: {len(wfo_df)}')

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
if len(wfo_df) > 0:
    axes[0].plot(wfo_df['val_start_epoch'], wfo_df['val_auc'], marker='o', linewidth=1.5)
    axes[0].axhline(0.5, color='red', linestyle='--', label='AUC=0.5 (aleatório)')
    axes[0].set_title('WFO — AUC-ROC por Janela de Validação')
    axes[0].set_ylabel('AUC-ROC')
    axes[0].legend()

    axes[1].plot(wfo_df['val_start_epoch'], wfo_df['val_winrate'] * 100, marker='s', color='green', linewidth=1.5)
    axes[1].set_title('WFO — Win Rate por Janela')
    axes[1].set_ylabel('Win Rate (%)')
    axes[1].set_xlabel('Epoch de início da validação')

plt.tight_layout()
plt.savefig('logs/wfo_analysis.png', dpi=110)
plt.show()

if len(wfo_df.dropna()) > 0:
    mean_auc = wfo_df['val_auc'].mean()
    print(f'AUC médio WFO: {mean_auc:.4f}')
    if mean_auc > 0.55:
        print('✅  Estratégia mantém edge no WFO.')
    else:
        print('❌  Edge não se mantém no WFO — possível overfitting.')

## 5. Otimização do take_profit_percent — Sweet Spot

In [ ]:
# ---------------------------------------------------------------------------
# Simula diferentes take_profit_percent usando os dados shadow
# ---------------------------------------------------------------------------

import sys
sys.path.insert(0, '.')

from backtest import load_ticks, run_accumulator_backtest
from strategy import AccumulatorStrategyConfig

TICK_FILES = list(Path('data').glob('*.csv'))
TICK_FILES = [f for f in TICK_FILES if 'shadow' not in f.name]

if not TICK_FILES:
    print('⚠️  Nenhum arquivo de ticks encontrado em data/. Pulando otimização de take_profit.')
else:
    tick_file = TICK_FILES[0]
    print(f'Usando arquivo: {tick_file}')
    ticks = load_ticks(tick_file)
    print(f'Ticks carregados: {len(ticks):,}')

    tp_values = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
    tp_results = []

    base_config = AccumulatorStrategyConfig()

    for tp in tp_values:
        result = run_accumulator_backtest(
            ticks=ticks,
            initial_balance=1000.0,
            stake=1.0,
            growth_rate=0.03,
            take_profit_percent=tp,
            barrier_percent=0.05,
            max_hold_ticks=10,
            cooldown_ticks=3,
            strategy_config=base_config,
        )
        tp_results.append({
            'take_profit_pct': tp,
            'trades': result['total_trades'],
            'winrate': result['winrate'],
            'net_profit': result['net_profit'],
            'max_drawdown_pct': result['max_drawdown_pct'],
            'max_loss_streak': result['max_loss_streak'],
        })

    tp_df = pd.DataFrame(tp_results)
    print(tp_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 5))
    ax2 = ax.twinx()
    ax.bar(tp_df['take_profit_pct'], tp_df['winrate'], alpha=0.6, label='Win Rate (%)')
    ax2.plot(tp_df['take_profit_pct'], tp_df['net_profit'], 'r-o', label='Net Profit')
    ax.set_xlabel('take_profit_percent')
    ax.set_ylabel('Win Rate (%)')
    ax2.set_ylabel('Lucro Líquido (R$)')
    ax.set_title('Otimização de take_profit_percent — Sweet Spot')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig('logs/tp_optimization.png', dpi=110)
    plt.show()

    best_row = tp_df.loc[tp_df['net_profit'].idxmax()]
    print(f'\nSweet spot: take_profit_percent = {best_row["take_profit_pct"]}%')
    print(f'Win rate: {best_row["winrate"]:.1f}% | Net profit: R$ {best_row["net_profit"]:.2f}')

## 6. Simulação com Latência (Slippage) — Impacto no Resultado

In [ ]:
# ---------------------------------------------------------------------------
# Compara backtest sem slippage vs com 1 e 2 ticks de atraso
# ---------------------------------------------------------------------------

if TICK_FILES:
    slippage_results = []
    for slip in [0, 1, 2]:
        result = run_accumulator_backtest(
            ticks=ticks,
            initial_balance=1000.0,
            stake=1.0,
            growth_rate=0.03,
            take_profit_percent=3.0,
            barrier_percent=0.05,
            max_hold_ticks=10,
            cooldown_ticks=3,
            strategy_config=base_config,
            slippage_ticks=slip,
        )
        slippage_results.append({
            'slippage_ticks': slip,
            'trades': result['total_trades'],
            'winrate': result['winrate'],
            'net_profit': result['net_profit'],
            'max_drawdown_pct': result['max_drawdown_pct'],
        })

    slip_df = pd.DataFrame(slippage_results)
    print('=== Impacto da Latência (Slippage) ===')
    print(slip_df.to_string(index=False))

    # Se a estratégia quebra com 1 tick de slippage, ela é frágil
    if len(slip_df) >= 2:
        wr_no_slip = slip_df.iloc[0]['winrate']
        wr_1_slip  = slip_df.iloc[1]['winrate']
        delta = wr_no_slip - wr_1_slip
        print(f'\nDelta win rate (0→1 tick slippage): {delta:.2f}%')
        if delta > 5:
            print('⚠️  Estratégia sensível à latência: perde >5pp com 1 tick de atraso.')
        else:
            print('✅  Estratégia robusta à latência.')

## 7. Tamanho de Amostra — Significância Estatística

In [ ]:
# ---------------------------------------------------------------------------
# Intervalo de confiança do win rate com Wilson score
# ---------------------------------------------------------------------------

from scipy.stats import norm

n_trades = len(df_clean)
p_hat = df_clean['y'].mean()
z = norm.ppf(0.975)  # 95% CI

# Wilson score interval
center = (p_hat + z**2 / (2*n_trades)) / (1 + z**2 / n_trades)
margin = z * np.sqrt(p_hat*(1-p_hat)/n_trades + z**2/(4*n_trades**2)) / (1 + z**2/n_trades)

ci_low = center - margin
ci_high = center + margin

print(f'n = {n_trades} trades')
print(f'Win rate: {p_hat*100:.2f}%')
print(f'IC 95% (Wilson): [{ci_low*100:.2f}%, {ci_high*100:.2f}%]')

# Break-even para ACCU 3% com payoff assimétrico
# 1 Loss = 1 stake; 1 Win ≈ 0.03 * stake → break-even = 1/(1+0.03) ≈ 97.1%
breakeven = 1.0 / (1.0 + 0.03)
print(f'\nBreak-even win rate (ACCU 3%): {breakeven*100:.1f}%')

n_needed = 1500
if n_trades < n_needed:
    print(f'⚠️  Amostra insuficiente: {n_trades} < {n_needed} trades mínimos para significância.')
    print(f'   Faltam {n_needed - n_trades} trades para atingir o limiar institucional.')
else:
    print(f'✅  Amostra suficiente ({n_trades} trades).')

## 8. Checklist de Aprovação — Critérios Go-Live

In [ ]:
# ---------------------------------------------------------------------------
# Checklist automático de aprovação
# ---------------------------------------------------------------------------

checklist = {}

# 1. Amostra mínima
checklist['1. >= 1500 trades validados'] = n_trades >= 1500

# 2. Expectância positiva
checklist['2. Expectância matemática > 0'] = expectancy > 0

# 3. WFO AUC médio > 0.55
if len(wfo_df.dropna()) > 0:
    checklist['3. WFO AUC médio > 0.55'] = wfo_df['val_auc'].mean() > 0.55
else:
    checklist['3. WFO AUC médio > 0.55 (dados insuf.)'] = False

# 4. IC 95% acima do break-even
checklist['4. IC 95% lower bound > break-even'] = ci_low > breakeven

# 5. Sem correlação serial suspeita
checklist['5. Sem correlação serial significativa'] = p_value >= 0.05

print('=' * 50)
print('CHECKLIST DE APROVAÇÃO GO-LIVE')
print('=' * 50)
for criterion, passed in checklist.items():
    icon = '✅' if passed else '❌'
    print(f'{icon}  {criterion}')

n_passed = sum(checklist.values())
n_total = len(checklist)
print('=' * 50)
print(f'Aprovados: {n_passed}/{n_total}')
if n_passed == n_total:
    print('🚀  ESTRATÉGIA APROVADA PARA GO-LIVE!')
else:
    print('🔬  Continue coletando dados e refinando antes do go-live.')